# Vígil.ia — RF-DETR **Small** COMPLETO → candidato pro Jetson Orin Nano

Pipeline inteiro num só notebook, mesmo caminho que consagrou o 11s/11x:

1. **Estágio 1 — base 12,5k:** RF-DETR Small partindo do COCO, treinado no dataset
   Roboflow (12.528 imagens, classificação → detecção via pseudo-rótulo Otsu).
2. **FT1 — fotos reais originais** (+ frames de vídeo de defeito) com cenas sintéticas.
3. **FT2 — suas capturas** (`soja pra treino`), caixa apertada.
4. **Vídeo:** `teste_soja.mp4` em 2 passadas (voto → veredito travado → render
   suavizado), mesma regra exigente do `vigil_deck.py`.
5. **Export ONNX** → engine TensorRT se gera depois, no próprio Jetson.

## Por que o Small

RF-DETR Small: **32,1M params, resolução 512, AP50 72,1 no COCO, 3,5 ms na T4
(TensorRT FP16), Apache 2.0**. Nano→Large têm quase os mesmos parâmetros (~30-34M;
o que muda é a resolução) — o Small é o meio-termo: mais detalhe que o Nano (512 vs
384, importa com grão encostado/sobreposto) e ainda com folga larga no Orin Nano
(referência: Nano a ~120 fps FP16 num Orin NX via TensorRT).

## v2 — o que mudou em relação à primeira rodada

A rodada v1 terminou com o mAP de val do FT2 em 0,43 (contra 0,92 no FT1), o que
parecia colapso mas **era a régua**, não o modelo: no vídeo real o FT2 até foi
melhor que o FT1. Três consertos:

1. **Val misto nos dois fine-tunes** (cenas multi-grão *e* fotos de 1 grão). Na v1
   o FT1 validava só com 1 grão e o FT2 só com cenas — inversão de tarefa entre
   estágios, que castiga DETR (ele aprende um prior de quantos objetos existem).
2. **Pool de recortes balanceado por classe** no train E no val. Na v1 só o train
   era balanceado, então o val do FT2 saiu com 39% skin-damaged contra 18% no
   train (e immature com 128 caixas de pouquíssimos grãos únicos).
3. **FT2 com `lr=1e-4` e paciência 15** (era 5e-5/8). Na v1 ele parou na época 12
   com o melhor na 4 — não teve chance de adaptar à mudança de distribuição.

Saídas em `_v2`, então os checkpoints da v1 ficam intactos para comparação.

## ⏱️ Tempo estimado (Colab **A100 40GB**, batch 16)

| Etapa | Tempo |
|---|---|
| Construção dos datasets (Otsu 12,5k + v3 + capturas) | ~25-40 min |
| Estágio 1 — base 12,5k (30 épocas) | **~2h30-3h30** |
| FT1 — fotos reais (40 épocas, early stopping) | ~1h-1h30 |
| FT2 — capturas (40 épocas, early stopping) | ~40min-1h |
| Vídeo (2 passadas) + export ONNX | ~10-15 min |
| **Total** | **~4h30-6h30** (tipicamente ~5h) |

- O RF-DETR treina mais devagar por imagem que o YOLO (backbone DINOv2 + cabeça
  DETR) — o estágio 1 é o grosso do custo.
- O early stopping costuma encurtar FT1/FT2.
- **Cada estágio salva no Drive e PULA se já existe** — dá pra dividir em 2-3
  sessões de Colab sem perder nada.
- Na T4: 4-6× mais lento (batch 4 + accum 4) — não recomendo, use A100.

> Honestidade (igual sempre): caixas de treino são pseudo-rótulo Otsu + sintético,
> não anotação humana. O juiz é o vídeo real. A latência medida aqui (GPU do Colab)
> é só **indicativa** — o número que decide é o TensorRT **no Jetson**.

## 0. Setup

In [ ]:
!pip -q install "rfdetr[train,loggers]" supervision
from importlib.metadata import version
print('rfdetr', version('rfdetr'), '| supervision', version('supervision'))

## 1. Config — caminhos e saídas no Drive

In [ ]:
import os, glob, shutil
import torch
from google.colab import drive
drive.mount('/content/drive')

# ===== RF-DETR Small: base 12,5k -> FT1 (fotos reais) -> FT2 (capturas) -> vídeo =====
SIZE = 640                          # letterbox do dataset (o rfdetr redimensiona
                                    # internamente p/ 512, a resolução do Small)
FRAME_STRIDE = 5
EXCLUIR_DO_TREINO = {'intact'}      # intact NUNCA treina via vídeo -> avaliação limpa
N_SYNTH = 1200
N_VAL_SCENES = 120
BLUR_FRAC = 0.4
VAL_FRAC = 0.15

# rfdetr recomenda batch TOTAL 16: A100 -> 16x1; T4 -> 4x4
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH, GRAD_ACCUM = (16, 1) if VRAM_GB > 30 else (4, 4)
print(f'GPU: {torch.cuda.get_device_name(0)} ({VRAM_GB:.0f} GB) -> batch {BATCH} x accum {GRAD_ACCUM}')

# --- 12,5k Roboflow (estágio 1) ---
CLS_BASE_CANDS = [
    '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder',
    '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder (Unzipped Files)',
]
CLS_BASE = next((p for p in CLS_BASE_CANDS if os.path.isdir(p)), None)
assert CLS_BASE, 'dataset 12,5k não encontrado:\n  ' + '\n  '.join(CLS_BASE_CANDS)

# --- fotos reais originais (FT1) ---
REAL_SRCS = [p for p in [
    '/content/drive/MyDrive/Soja total/Soja total/Lotes',
    '/content/drive/MyDrive/Soja pra completar',
] if os.path.isdir(p)]
HAS_FT1 = len(REAL_SRCS) > 0

# --- vídeos de defeito p/ enriquecer o FT1 (opcional) ---
VAL_ROOT = '/content/drive/MyDrive/Vídeos para treino/Treino'
HAS_VAL_ROOT = os.path.isdir(VAL_ROOT)

# --- suas capturas revisadas (FT2) ---
CAP_SRCS = ['/content/drive/MyDrive/soja pra treino']
for p in CAP_SRCS:
    assert os.path.isdir(p), f'capturas não encontradas: {p}'

# saídas no Drive — cada estágio salva e PULA se já existe (resumível)
# v2 = val misto + pools de recorte balanceados + receita do FT2 corrigida.
# O ESTÁGIO 1 NÃO MUDOU (mesmo dataset base, mesma receita) -> reaproveita o
# .pth da rodada v1 se ele já estiver no Drive, economizando ~2h de A100.
STAGE1_PTH = '/content/drive/MyDrive/soja_rfdetr_small_stage1.pth'
FT1_PTH    = '/content/drive/MyDrive/soja_rfdetr_small_ft1_v2.pth'
FINAL_PTH  = '/content/drive/MyDrive/soja_rfdetr_small_final_v2.pth'

print('12,5k :', CLS_BASE)
print('FT1   :', REAL_SRCS if HAS_FT1 else 'PULADO (fotos reais originais não achadas)')
print('vídeos:', VAL_ROOT if HAS_VAL_ROOT else 'PULADO')
print('FT2   :', CAP_SRCS)

## 2. Funções de dataset

Idênticas ao `treino_11s_completo_640.ipynb` (pseudo-rótulo Otsu, recorte com máscara
limpa/erodida, cenas multi-grão com borda crispa). A única peça nova é o
`yolo_to_coco` no fim: o `rfdetr` treina com **COCO JSON** (`_annotations.coco.json`
por split), não com os `.txt` estilo YOLO — então geramos o dataset igual sempre e
convertemos o layout no final (imagens entram por hardlink, sem duplicar disco).

In [ ]:
import glob, hashlib, unicodedata, cv2, yaml, json
import numpy as np
from PIL import Image

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
RNG = np.random.default_rng(42)

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

def collect_real(srcs, val_frac=0.15):
    items = []
    for src in srcs:
        for root, _, files in os.walk(src):
            cls = None
            for part in reversed(root.split(os.sep)):
                c = class_of(part)
                if c is not None:
                    cls = c; break
            if cls is None:
                continue
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    p = os.path.join(root, fn)
                    h = int(hashlib.md5(p.encode()).hexdigest(), 16)
                    items.append((p, cls, 'val' if (h % 100) < val_frac * 100 else 'train'))
    from collections import Counter
    print('coletado:', dict(Counter(sp for _, _, sp in items)))
    return items

def sat_box(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def otsu_box(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def letterbox640(img, size=640):
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top

def motion_blur(img, rng=RNG):
    k = int(rng.choice([7, 9, 11, 13, 15]))
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= max(kernel.sum(), 1e-6)
    return cv2.filter2D(img, -1, kernel)

def extract_cutout(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    bx, by, bw, bh = cv2.boundingRect(c)
    # solidez: grão é compacto; blob esfarrapado daria caixa larga -> rejeita
    if area < 0.55 * bw * bh:
        return None
    mask = np.zeros((h, w), np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    # erode 1px: tira o halo -> a caixa cola no grão
    mask = cv2.erode(mask, np.ones((3, 3), np.uint8))
    ys, xs = np.where(mask > 0)
    if not len(xs):
        return None
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    return img[y0:y1, x0:x1], mask[y0:y1, x0:x1]

def make_scene(cutouts, rng=RNG, size=640):
    bg = int(rng.integers(20, 130))
    canvas = np.clip(np.full((size, size, 3), bg, np.int16)
                     + rng.normal(0, 6, (size, size, 3)), 0, 255).astype(np.uint8)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    lo, hi = int(60 * size / 640), int(150 * size / 640)
    for _ in range(int(rng.integers(6, 26))):
        cls, crop, mask = cutouts[int(rng.integers(len(cutouts)))]
        s = int(rng.integers(lo, hi)) / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        placed = False
        for _try in range(20):
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            if inter.sum() <= 0.15 * (mask3 > 0).sum():
                placed = True
                break
        if not placed:
            continue
        alpha = (cv2.GaussianBlur(mask3, (5, 5), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size, gw / size, gh / size))
    return canvas, boxes

def balance_train(items):
    from collections import defaultdict
    train = [it for it in items if it[2] == 'train']
    rest = [it for it in items if it[2] != 'train']
    by = defaultdict(list)
    for it in train:
        by[it[1]].append(it)
    mx = max(len(v) for v in by.values())
    out = []
    for c, v in by.items():
        out += v + [v[int(i)] for i in RNG.integers(0, len(v), mx - len(v))]
    print('balanceado (train):', {NAMES[c]: sum(1 for it in out if it[1] == c) for c in sorted(by)})
    return out + rest

def balance_cutouts(cuts, rng):
    """Iguala o nº de recortes por classe no pool.

    O make_scene sorteia UNIFORME do pool, então pool torto => cenas tortas.
    Foi exatamente isso que fez o val do FT2 ter 39% skin-damaged contra 18% no
    train (e o immature virar 128 caixas de pouquíssimos grãos únicos), o que
    derrubou o mAP de val sem o modelo ter piorado.
    """
    from collections import defaultdict
    by = defaultdict(list)
    for c in cuts:
        by[c[0]].append(c)
    if not by:
        return cuts
    mx = max(len(v) for v in by.values())
    out = []
    for c, v in by.items():
        out += v + [v[int(i)] for i in rng.integers(0, len(v), mx - len(v))]
    print('   pool de recortes balanceado:',
          {NAMES[c]: sum(1 for x in out if x[0] == c) for c in sorted(by)})
    return out


def write_scenes(out_dir, cutouts, split, n, rng, blur_frac):
    """Escreve n cenas multi-grão sintéticas em out_dir/images/<split>."""
    made = 0
    for j in range(n):
        if j % 200 == 0:
            print(f'  cenas {split} {j}/{n}…', flush=True)
        canvas, boxes = make_scene(cutouts, rng=rng)
        if not boxes:
            continue
        if rng.random() < blur_frac:
            canvas = motion_blur(canvas, rng=rng)
        stem = f'synth_{split}_{j:05d}'
        cv2.imwrite(f'{out_dir}/images/{split}/{stem}.jpg', canvas,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{split}/{stem}.txt', 'w').write(
            '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}'
                         for c, cx, cy, w, h in boxes))
        made += 1
    print(f'cenas {split}: {made}')


def write_single(img, cls, out_dir, split, stem, with_blur, max_area=None):
    """Escreve 1 foto de grão solto (letterbox 640) + cópia borrada se pedido.

    max_area corta o vício "caixa = quadro inteiro" (usado no FT2, onde as
    capturas do vigil_deck são recortes apertados).
    """
    box = sat_box(img) or otsu_box(img)
    if box is None:
        return 0
    if max_area is not None and box[2] * box[3] > max_area:
        return 0
    h0, w0 = img.shape[:2]
    lb, s, left, top = letterbox640(img)
    cx, cy, ww, hh = box
    cx = (cx * w0 * s + left) / 640.0
    cy = (cy * h0 * s + top) / 640.0
    ww = (ww * w0 * s) / 640.0
    hh = (hh * h0 * s) / 640.0
    line = f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}'
    cv2.imwrite(f'{out_dir}/images/{split}/{stem}.jpg', lb,
                [cv2.IMWRITE_JPEG_QUALITY, 95])
    open(f'{out_dir}/labels/{split}/{stem}.txt', 'w').write(line)
    if not with_blur:
        return 1
    cv2.imwrite(f'{out_dir}/images/{split}/{stem}b.jpg', motion_blur(lb),
                [cv2.IMWRITE_JPEG_QUALITY, 95])
    open(f'{out_dir}/labels/{split}/{stem}b.txt', 'w').write(line)
    return 2


def build_v3(items, out_dir, n_synth=600, n_val_scenes=N_VAL_SCENES,
             blur_frac=0.4):
    """FT1: fotos reais soltas + cenas multi-grão.

    VAL MISTO (single + cena), igual ao train. Sem isso o FT1 seleciona o melhor
    checkpoint olhando SÓ foto de 1 grão, e o FT2 (100% cena) vira uma inversão
    de tarefa — o que castiga a família DETR, que aprende um prior de quantos
    objetos existem no quadro.
    """
    assert items, 'Nenhuma imagem coletada! Confira REAL_SRCS.'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)
    cut_train, cut_val = [], []
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  fotos {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1
            continue
        cut = extract_cutout(img)
        if cut is not None:
            (cut_train if sp == 'train' else cut_val).append((cls, cut[0], cut[1]))
        n = write_single(img, cls, out_dir, sp, f'{sp}_{i:06d}',
                         with_blur=(sp == 'train'))
        kept += n
        if not n:
            skipped += 1
    print(f'fotos reais: kept={kept} skipped={skipped} | '
          f'recortes train: {len(cut_train)} val: {len(cut_val)}')
    assert cut_train, 'Nenhum recorte extraído!'
    cut_train = balance_cutouts(cut_train, np.random.default_rng(7))
    write_scenes(out_dir, cut_train, 'train', n_synth,
                 np.random.default_rng(42), blur_frac)
    if cut_val:
        cut_val = balance_cutouts(cut_val, np.random.default_rng(8))
        write_scenes(out_dir, cut_val, 'val', n_val_scenes,
                     np.random.default_rng(123), blur_frac)
    else:
        print('AVISO: sem recortes de val -> val do FT1 fica só com singles')

SPLIT_MAP = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}

def collect_base(base_dir):
    """Acha train/valid/test em qualquer profundidade dentro do dataset 12,5k."""
    items = []
    for root, dirs, _ in os.walk(base_dir):
        for d in list(dirs):
            sp = SPLIT_MAP.get(d.lower())
            if sp is None:
                continue
            split_dir = os.path.join(root, d)
            for folder in sorted(os.listdir(split_dir)):
                cls = class_of(folder)
                if cls is None:
                    continue
                for p_ in glob.glob(os.path.join(split_dir, folder, '*')):
                    if p_.lower().endswith(IMG_EXT):
                        items.append((p_, cls, sp))
            dirs.remove(d)
    from collections import Counter
    print('coletado (base 12,5k):', dict(Counter(sp for _, _, sp in items)))
    return items

def build_base(items, out_dir):
    """Base de detecção: pseudo-rótulo Otsu (1 grão/img, fundo preto) — idêntico
    ao estágio base do RT-DETR/YOLO11x."""
    assert items, 'Nenhuma imagem do 12,5k coletada!'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 1000 == 0:
            print(f'  base {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        box = otsu_box(img)
        if box is None:
            skipped += 1; continue
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(
            f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}')
        kept += 1
    print(f'base: kept={kept} skipped={skipped}')

def build_ft(items, out_dir, n_synth=N_SYNTH, n_val_scenes=N_VAL_SCENES,
             blur_frac=BLUR_FRAC):
    """FT2: capturas revisadas, caixa apertada.

    VAL MISTO (single + cena) e pools de recorte balanceados por classe nos DOIS
    splits — o val v1 tinha distribuição de classe bem diferente do train, então
    o "melhor checkpoint" era escolhido por uma régua torta.
    """
    assert items, 'Nenhuma imagem coletada! Confira as fontes.'
    shutil.rmtree(out_dir, ignore_errors=True)
    for sp in ('train', 'val'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)
    cut_train, cut_val = [], []
    n_single = {'train': 0, 'val': 0}
    skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  fotos {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1
            continue
        cut = extract_cutout(img)
        if cut is not None:
            (cut_train if sp == 'train' else cut_val).append((cls, cut[0], cut[1]))
        # max_area=0.75: recorte apertado (grão = quadro) NÃO entra como single,
        # senão ensina o vício "caixa = quadro" que derrubou o base_12k
        n_single[sp] += write_single(img, cls, out_dir, sp, f'single_{i:06d}',
                                     with_blur=(sp == 'train'), max_area=0.75)
    print(f'singles c/ caixa real: {n_single} | skipped: {skipped} | '
          f'recortes train: {len(cut_train)} val: {len(cut_val)}')
    assert cut_train, 'Nenhum recorte de treino extraído!'
    assert cut_val, ('Nenhum recorte de VAL extraído — sem como montar o val '
                     'multi-grão. Confira as imagens do split val.')
    cut_train = balance_cutouts(cut_train, np.random.default_rng(7))
    cut_val = balance_cutouts(cut_val, np.random.default_rng(8))
    write_scenes(out_dir, cut_train, 'train', n_synth,
                 np.random.default_rng(42), blur_frac)
    write_scenes(out_dir, cut_val, 'val', n_val_scenes,
                 np.random.default_rng(123), blur_frac)


# ---- a peça nova: YOLO txt -> layout COCO do rfdetr ----
def yolo_to_coco(split_srcs, out_dir, names=NAMES):
    """split_srcs: {'train': [(img_dir, lbl_dir), ...], 'valid': [...], 'test': [...]}.
    Gera out_dir/{split}/_annotations.coco.json com as imagens hardlinkadas."""
    shutil.rmtree(out_dir, ignore_errors=True)
    cats = [{'id': i + 1, 'name': n, 'supercategory': 'soja'} for i, n in enumerate(names)]
    for split, srcs in split_srcs.items():
        os.makedirs(f'{out_dir}/{split}', exist_ok=True)
        images, anns = [], []
        img_id = ann_id = 0
        for img_dir, lbl_dir in srcs:
            for p in sorted(glob.glob(os.path.join(img_dir, '*.jpg'))):
                stem = os.path.splitext(os.path.basename(p))[0]
                lp = os.path.join(lbl_dir, stem + '.txt')
                if not os.path.exists(lp):
                    continue
                with Image.open(p) as im:
                    w, h = im.size
                fn = f'{img_id:06d}_{os.path.basename(p)}'
                dst = f'{out_dir}/{split}/{fn}'
                try:
                    os.link(p, dst)
                except OSError:
                    shutil.copy(p, dst)
                images.append({'id': img_id, 'file_name': fn, 'width': w, 'height': h})
                for line in open(lp):
                    parts = line.split()
                    if len(parts) != 5:
                        continue
                    c = int(parts[0])
                    cx, cy, bw, bh = map(float, parts[1:])
                    x, y = (cx - bw / 2) * w, (cy - bh / 2) * h
                    anns.append({'id': ann_id, 'image_id': img_id, 'category_id': c + 1,
                                 'bbox': [round(x, 2), round(y, 2),
                                          round(bw * w, 2), round(bh * h, 2)],
                                 'area': round(bw * w * bh * h, 2), 'iscrowd': 0})
                    ann_id += 1
                img_id += 1
        json.dump({'images': images, 'annotations': anns, 'categories': cats},
                  open(f'{out_dir}/{split}/_annotations.coco.json', 'w'))
        print(f'{out_dir}/{split}: {len(images)} imgs, {len(anns)} caixas')
    return out_dir

print('funções prontas')


## 3. Constrói base 12,5k (estágio 1) + fotos reais (FT1) — ~25-40 min total

In [ ]:
# ---- estágio 1: base 12,5k (pseudo-rótulo Otsu) -> COCO ----
BASE_YOLO = '/content/soja_det_base'
if not os.path.isdir(f'{BASE_YOLO}/images/train'):
    print('construindo base 12,5k (~10 min)…')
    build_base(collect_base(CLS_BASE), BASE_YOLO)
BASE_COCO = '/content/coco_base'
if not os.path.isdir(f'{BASE_COCO}/train'):
    yolo_to_coco({'train': [(f'{BASE_YOLO}/images/train', f'{BASE_YOLO}/labels/train')],
                  'valid': [(f'{BASE_YOLO}/images/val',   f'{BASE_YOLO}/labels/val')],
                  'test':  [(f'{BASE_YOLO}/images/test',  f'{BASE_YOLO}/labels/test')]},
                 BASE_COCO)
print('base 12,5k COCO:', BASE_COCO)

# ---- FT1 (parte 1): v3 fotos reais (real + blur + cenas sintéticas), formato YOLO ----
V3 = '/content/soja_det_v3_v2'
if HAS_FT1 and not os.path.isdir(f'{V3}/images/train'):
    print('construindo v3 fotos reais (~10-15 min)…')
    build_v3(collect_real(REAL_SRCS), V3)

### 3b. Frames de vídeo de defeito para o FT1 (opcional)

In [ ]:
if not HAS_VAL_ROOT:
    HAS_VIDEO_DATA = False
    print('vídeos de defeito: PULADO (VAL_ROOT ausente)')
else:
    VIDEO_EXT = ('.mp4', '.mov', '.avi', '.mkv')
    ROI_BOTTOM = 0.15
    MIN_SAT    = 40

    def video_frames(path, stride=FRAME_STRIDE):
        cap = cv2.VideoCapture(path)
        k = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if k % stride == 0:
                yield k, frame[:int(frame.shape[0] * (1 - ROI_BOTTOM))]
            k += 1
        cap.release()

    def _dedupe(boxes):
        out = []
        boxes = sorted(boxes, key=lambda b: -(b[2] * b[3]))
        for b in boxes:
            bx1, by1, bx2, by2 = b[0]-b[2]/2, b[1]-b[3]/2, b[0]+b[2]/2, b[1]+b[3]/2
            dup = False
            for kk in out:
                kx1, ky1, kx2, ky2 = kk[0]-kk[2]/2, kk[1]-kk[3]/2, kk[0]+kk[2]/2, kk[1]+kk[3]/2
                iw = max(0, min(bx2, kx2) - max(bx1, kx1))
                ih = max(0, min(by2, ky2) - max(by1, ky1))
                if iw * ih > 0.4 * (b[2] * b[3]):
                    dup = True; break
            if not dup:
                out.append(b)
        return out

    def multi_boxes(img, min_frac=0.0015, max_frac=0.05, edge=0.02):
        h, w = img.shape[:2]
        S = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[:, :, 1]
        blur = cv2.GaussianBlur(S, (5, 5), 0)
        _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
        cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        boxes = []
        for c in cnts:
            a = cv2.contourArea(c)
            if not (min_frac * h * w <= a <= max_frac * h * w):
                continue
            x, y, bw, bh = cv2.boundingRect(c)
            if bw / max(bh, 1) > 2.2 or bh / max(bw, 1) > 2.2:
                continue
            if a < 0.5 * bw * bh:
                continue
            if S[y:y+bh, x:x+bw].mean() < MIN_SAT:
                continue
            cx, cy = (x + bw/2) / w, (y + bh/2) / h
            if not (edge < cx < 1-edge and edge < cy < 1-edge):
                continue
            pad = int(0.04 * min(bw, bh)) + 2
            x1, y1 = max(0, x - pad), max(0, y - pad)
            x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
            boxes.append((((x1+x2)/2)/w, ((y1+y2)/2)/h, (x2-x1)/w, (y2-y1)/h))
        return _dedupe(boxes)

    VID = '/content/soja_video_640'
    shutil.rmtree(VID, ignore_errors=True)
    for sp in ('train', 'val'):
        os.makedirs(f'{VID}/images/{sp}', exist_ok=True)
        os.makedirs(f'{VID}/labels/{sp}', exist_ok=True)

    stats = {}
    for entry in sorted(os.listdir(VAL_ROOT)):
        sub = os.path.join(VAL_ROOT, entry)
        if not os.path.isdir(sub):
            continue
        cls = class_of(entry)
        if cls is None:
            continue
        if NAMES[cls] in EXCLUIR_DO_TREINO:
            print(f'{entry} -> {NAMES[cls]}: EXCLUÍDO do treino (avaliação limpa)')
            continue
        vids = [os.path.join(sub, f) for f in sorted(os.listdir(sub))
                if f.lower().endswith(VIDEO_EXT)]
        if not vids:
            continue
        val_video = vids[-1] if len(vids) >= 2 else None
        wrote = {'train': 0, 'val': 0}
        for vi, vp in enumerate(vids):
            frames = list(video_frames(vp))
            if val_video is None:
                cut = int(0.85 * len(frames))
                split_of = lambda i: ('train' if i < cut else
                                      None if i < cut + 150 // FRAME_STRIDE else 'val')
            else:
                split_of = lambda i: 'val' if vp == val_video else 'train'
            for i, (k, frame) in enumerate(frames):
                sp = split_of(i)
                if sp is None:
                    continue
                boxes = multi_boxes(frame)
                if not (3 <= len(boxes) <= 80):
                    continue
                lb, s, left, top = letterbox640(frame, size=SIZE)
                H0, W0 = frame.shape[:2]
                lines = []
                for cx, cy, ww, hh in boxes:
                    cx2 = (cx * W0 * s + left) / SIZE
                    cy2 = (cy * H0 * s + top) / SIZE
                    lines.append(f'{cls} {cx2:.6f} {cy2:.6f} {ww*W0*s/SIZE:.6f} {hh*H0*s/SIZE:.6f}')
                stem = f'{NAMES[cls]}_{vi}_{k:05d}'
                cv2.imwrite(f'{VID}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
                open(f'{VID}/labels/{sp}/{stem}.txt', 'w').write('\n'.join(lines))
                wrote[sp] += 1
        stats[NAMES[cls]] = wrote
        print(f'{entry} -> {NAMES[cls]}: {wrote}')

    HAS_VIDEO_DATA = any(v['train'] for v in stats.values())
    print('\nvídeo no fine-tune:', stats if HAS_VIDEO_DATA else
          'NENHUM — FT1 será só fotos reais')

In [ ]:
# ---- FT1 (parte 2): junta v3 + frames de vídeo num único dataset COCO ----
FT1_COCO = None
if HAS_FT1:
    FT1_COCO = '/content/coco_ft1_v2'
    if not os.path.isdir(f'{FT1_COCO}/train'):
        train_srcs = [(f'{V3}/images/train', f'{V3}/labels/train')]
        if HAS_VIDEO_DATA:
            train_srcs.append((f'{VID}/images/train', f'{VID}/labels/train'))
        val_srcs = [(f'{V3}/images/val', f'{V3}/labels/val')]
        yolo_to_coco({'train': train_srcs, 'valid': val_srcs, 'test': val_srcs}, FT1_COCO)
    print('FT1 COCO:', FT1_COCO)
else:
    print('FT1 pulado — o FT2 parte direto do estágio 1')

## 4. Constrói o dataset das SUAS capturas (FT2, caixa apertada)

In [ ]:
CAP_OUT = '/content/soja_cap_v2'
if not os.path.isdir(f'{CAP_OUT}/images/train'):
    build_ft(collect_real(CAP_SRCS), CAP_OUT, n_synth=N_SYNTH,
             n_val_scenes=N_VAL_SCENES, blur_frac=BLUR_FRAC)
FT2_COCO = '/content/coco_ft2_v2'
if not os.path.isdir(f'{FT2_COCO}/train'):
    yolo_to_coco({'train': [(f'{CAP_OUT}/images/train', f'{CAP_OUT}/labels/train')],
                  'valid': [(f'{CAP_OUT}/images/val', f'{CAP_OUT}/labels/val')],
                  'test':  [(f'{CAP_OUT}/images/val', f'{CAP_OUT}/labels/val')]},
                 FT2_COCO)
print('FT2 COCO:', FT2_COCO)

## 5. Treino em 3 estágios (Small do zero → final) — ~4h-6h na A100

Mesma lógica do 11s: cada estágio salva no Drive e **pula se já existe** (resumível —
se a sessão cair, rode tudo de novo que ele retoma de onde parou).

Diferenças de motor vs ultralytics:
- `pretrain_weights=` encadeia os estágios (pesos entram, otimizador zera — igual
  ao `YOLO(pt_anterior)`).
- `skip_best_epochs`: partindo de pesos já bons, a "melhor época" ingênua seria a
  época 0 (sem treinar nada) — esse parâmetro segura a seleção de best/paciência
  até o modelo adaptar. *(Se a sua versão do rfdetr não aceitar, remova.)*
- RF-DETR é NMS-free — nada de `agnostic_nms` no treino.

| Estágio | Épocas | LR | Paciência | Tempo (A100) |
|---|---|---|---|---|
| 1 — base 12,5k | 30 | 1e-4 | 8 | ~2h — **pula se o `.pth` da v1 existe** |
| FT1 — fotos reais | 40 | 1e-4 | 12 | ~1h-1h30 |
| FT2 — capturas | 40 | **1e-4** | **15** | ~1h-1h30 |

In [ ]:
from rfdetr import RFDETRSmall

# paciência agora é POR ESTÁGIO: o FT2 precisa de mais folga (na v1 ele
# parou na época 12 com o melhor na 4 — não teve chance de adaptar).
TRAIN_COMMON = dict(batch_size=BATCH, grad_accum_steps=GRAD_ACCUM,
                    early_stopping=True, tensorboard=False, wandb=False)

def best_ckpt(out_dir):
    p = os.path.join(out_dir, 'checkpoint_best_total.pth')
    assert os.path.exists(p), f'checkpoint_best_total.pth não apareceu em {out_dir}'
    return p

# ===== ESTÁGIO 1: COCO -> base 12,5k (~2h30-3h30 na A100) =====
if os.path.exists(STAGE1_PTH):
    print('estágio 1 já no Drive:', STAGE1_PTH)
else:
    print('ESTÁGIO 1: RF-DETR Small <- COCO na base 12,5k')
    out = '/content/runs_rfdetr/small_stage1'
    m1 = RFDETRSmall()
    m1.train(dataset_dir=BASE_COCO, epochs=30, lr=1e-4,
             early_stopping_patience=8, output_dir=out, **TRAIN_COMMON)
    shutil.copy(best_ckpt(out), STAGE1_PTH)
    print('salvo:', STAGE1_PTH)

# ===== FT1: fotos reais originais (+ vídeos de defeito) (~1h-1h30) =====
if not HAS_FT1:
    FT1_SRC = STAGE1_PTH
    print('FT1 pulado — FT2 parte do estágio 1')
elif os.path.exists(FT1_PTH):
    FT1_SRC = FT1_PTH
    print('FT1 já no Drive:', FT1_PTH)
else:
    print('FT1: fine-tune fotos reais')
    out = '/content/runs_rfdetr/small_ft1'
    m2 = RFDETRSmall(pretrain_weights=STAGE1_PTH)
    m2.train(dataset_dir=FT1_COCO, epochs=40, lr=1e-4,
             early_stopping_patience=12, skip_best_epochs=3,
             output_dir=out, **TRAIN_COMMON)
    shutil.copy(best_ckpt(out), FT1_PTH)
    FT1_SRC = FT1_PTH
    print('salvo:', FT1_PTH)

# ===== FT2: suas capturas (caixa apertada, multi-grão) (~40min-1h) =====
if os.path.exists(FINAL_PTH):
    print('FT2 (final) já no Drive:', FINAL_PTH)
else:
    print('FT2: fine-tune capturas ->', FINAL_PTH)
    out = '/content/runs_rfdetr/small_ft2'
    m3 = RFDETRSmall(pretrain_weights=FT1_SRC)
    m3.train(dataset_dir=FT2_COCO, epochs=40, lr=1e-4,
             early_stopping_patience=15, skip_best_epochs=3,
             output_dir=out, **TRAIN_COMMON)
    shutil.copy(best_ckpt(out), FINAL_PTH)
    print('MODELO FINAL salvo:', FINAL_PTH)

## 6. Juiz de verdade — `teste_soja.mp4`

2 passadas (votos → render com veredito final desde o 1º frame), caixas suavizadas,
mesma regra exigente do `vigil_deck.py`. O rastreamento é o **ByteTrack do
`supervision`** (mesmo contrato do `model.track()` do ultralytics: id persistente
por grão). Gera `comparativo_rfdetr_small.mp4` no Drive — assista lado a lado com
os `comparativo_*.mp4` dos YOLOs.

In [ ]:
import time
import torch
from collections import defaultdict, Counter
import supervision as sv

def first_existing(*paths):
    return next((p for p in paths if os.path.exists(p)), None)

VIDEO_TESTE = first_existing('/content/drive/MyDrive/teste_soja.mp4',
                             '/content/drive/MyDrive/teste_soja.avi')
assert VIDEO_TESTE, 'teste_soja.(mp4|avi) não encontrado no Drive!'

# regra exigente por classe (igual ao vigil_deck.py)
RATIOS = {'broken': 0.85, 'skin-damaged': 0.80, 'spotted': 0.75, 'immature': 0.75}
MIN_TRACK_FRAMES = 3
SMOOTH = 0.4
CONF = 0.30
PT_LABEL = {'broken': 'Quebrado', 'immature': 'Imaturo', 'intact': 'Intacto',
            'skin-damaged': 'Casca danif.', 'spotted': 'Manchado'}
COLORS = {'intact': (90, 200, 90), 'immature': (60, 200, 200), 'broken': (170, 100, 210),
          'skin-damaged': (255, 160, 60), 'spotted': (70, 70, 235)}

# O rfdetr devolve class_id na ORDEM de NAMES, mas a base (0 ou 1) varia.
# Mapear pelo category_id do JSON COCO (1..5, porque COCO e 1-indexed) desloca
# tudo em 1 e faz 'intact' sair como 'immature' — e 'spotted' nunca aparecer.
# Por isso a base e DETECTADA a partir dos ids realmente vistos no video.
def _base_de(ids):
    return 0 if min(ids) == 0 else (1 if max(ids) == 5 else 0)

def veredito(cnt):
    top, w = cnt.most_common(1)[0]
    if top == 'intact':
        return 'intact'
    return top if w >= RATIOS.get(top, 0.8) * sum(cnt.values()) else 'intact'

def coletar_rf(model, video_path, conf=CONF):
    """Passada 1: votos por grão no vídeo TODO + dets por frame (ByteTrack do sv)."""
    tracker = sv.ByteTrack()
    votes = defaultdict(Counter)
    seen = Counter()
    dets_per_frame = defaultdict(list)
    ids_vistos = set()
    cap = cv2.VideoCapture(video_path)
    k, t_inf = 0, 0.0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t0 = time.time()
        det = model.predict(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), threshold=conf)
        t_inf += time.time() - t0
        # paridade com o pipeline YOLO/RT-DETR: 1 caixa por grão entre classes
        det = det.with_nms(threshold=0.6, class_agnostic=True)
        det = tracker.update_with_detections(det)
        if det.tracker_id is not None:
            for (x1, y1, x2, y2), tid, cid, cf in zip(det.xyxy.astype(int),
                                                      det.tracker_id,
                                                      det.class_id,
                                                      det.confidence):
                ids_vistos.add(int(cid))
                votes[int(tid)][int(cid)] += float(cf)   # vota no ID CRU
                seen[int(tid)] += 1
                dets_per_frame[k].append((int(tid), int(x1), int(y1), int(x2), int(y2)))
        k += 1
    cap.release()
    assert ids_vistos, 'o modelo nao detectou nada no video — baixe CONF'
    base = _base_de(ids_vistos)
    print(f'    class_id vistos: {sorted(ids_vistos)} -> base={base}')
    def nome(c):
        i = c - base
        return NAMES[i] if 0 <= i < len(NAMES) else f'?{c}'
    verdict = {}
    for tid, v in votes.items():
        if seen[tid] < MIN_TRACK_FRAMES:
            continue
        verdict[tid] = veredito(Counter({nome(c): w for c, w in v.items()}))
    return dets_per_frame, verdict, seen, k, (t_inf / max(k, 1)) * 1000

def render(video_path, dets_per_frame, verdict, out_path):
    """Passada 2: desenha cada grão já com o veredito final, caixa suavizada (EMA)."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    smooth = {}
    writer, k = None, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
        for tid, x1, y1, x2, y2 in dets_per_frame.get(k, []):
            if tid not in verdict:
                continue
            if tid in smooth:
                px1, py1, px2, py2 = smooth[tid]
                x1 = int(SMOOTH * x1 + (1 - SMOOTH) * px1)
                y1 = int(SMOOTH * y1 + (1 - SMOOTH) * py1)
                x2 = int(SMOOTH * x2 + (1 - SMOOTH) * px2)
                y2 = int(SMOOTH * y2 + (1 - SMOOTH) * py2)
            smooth[tid] = (x1, y1, x2, y2)
            cls = verdict[tid]
            color = COLORS[cls]
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f'#{tid} {PT_LABEL[cls]}', (x1, max(18, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        writer.write(frame)
        k += 1
    cap.release(); writer.release()

assert os.path.exists(FINAL_PTH), 'treine antes (célula 5)'
print(f'>>> rfdetr_small: passada 1 (votos) em {os.path.basename(VIDEO_TESTE)}…')
m = RFDETRSmall(pretrain_weights=FINAL_PTH)
try:
    m.inference(dtype=torch.float16)   # FP16: o proprio rfdetr avisa ~8x de ganho
except Exception as e:
    print('    FP16 pulado (segue em FP32):', e)
dets, verdict, seen, n_frames, ms = coletar_rf(m, VIDEO_TESTE)
print('    passada 2 (render)…')
OUT_MP4 = '/content/drive/MyDrive/comparativo_rfdetr_small.mp4'
render(VIDEO_TESTE, dets, verdict, OUT_MP4)
dist = Counter(verdict.values())
print(f'    {n_frames} frames | {sum(dist.values())} grãos c/ veredito')
print(f'    {ms:.1f} ms/frame na GPU do Colab (INDICATIVO — o juiz de fps é o Jetson)')
print(f'    salvo: {OUT_MP4}')
print()
print('=== veredito por classe ===')
for cls in NAMES:
    print(f'  {PT_LABEL[cls]:14s} {dist.get(cls, 0):3d}')
print()
print('Assista comparativo_rfdetr_small.mp4 lado a lado com os comparativo_*.mp4')
print('dos YOLOs: menos caixa piscando/classe trocando/alucinação = melhor.')

## 7. Export ONNX (caminho pro Jetson)

In [ ]:
# ONNX do modelo final -> Drive. No Jetson, o engine TensorRT é gerado NO
# PRÓPRIO aparelho (o .engine é atado ao hardware/versão do TensorRT).
out = '/content/export_small'
os.makedirs(out, exist_ok=True)
m = RFDETRSmall(pretrain_weights=FINAL_PTH)
m.export(output_dir=out)
onnx = first_existing(f'{out}/inference_model.onnx', *glob.glob(f'{out}/*.onnx'))
assert onnx, f'export não gerou .onnx em {out}'
DST_ONNX = '/content/drive/MyDrive/soja_rfdetr_small_final.onnx'
shutil.copy(onnx, DST_ONNX)
print('ONNX no Drive ->', DST_ONNX)

## Depois — checklist do Jetson Orin Nano

Quando o Jetson chegar:

1. Copiar o `.onnx` pro Jetson e gerar o engine **nele**:
   ```bash
   /usr/src/tensorrt/bin/trtexec --onnx=soja_rfdetr_small_final.onnx \
       --saveEngine=soja_rfdetr_small_fp16.engine --fp16
   ```
2. Medir latência real (`trtexec` já reporta fps) e comparar com o YOLO
   equivalente exportado p/ TensorRT — decisão por dados, como sempre.
3. Integração: `infracv/rf-detr-cpp` (engine C++ TensorRT pronto, FP16/INT8,
   suporta Orin) ou RF-DETR + DeepStream (blog da Roboflow).
4. Se FP16 não bastar em fps, INT8 com calibração nas suas cenas é o próximo degrau.

**Critério de decisão** (mesmo espírito do tira-teima):
- Qualidade no `teste_soja.mp4` (caixa que cola, sem alucinação, `spotted` correto)
- fps real no Jetson em FP16
- Se perder feio pro 11s no vídeo → RF-DETR sai da infra do Jetson e registramos
  o resultado negativo, como fizemos com o LR discriminativo.